In [1]:
import pandas
import pyarrow
import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

try:
    SparkContext.getOrCreate().stop()
except:
    pass

In [3]:
# 1. Setup Spark
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("NYC-Taxi-Ingestion") \
    .getOrCreate()

# Print the Spark version to verify successful connection
print(f"Spark version: {spark.version}")

TypeError: 'JavaPackage' object is not callable

In [ ]:
output_dir = "/home/jovyan/work/data"
file_path = f"{output_dir}/fhvhv_Jan2025.parquet"

In [ ]:
df = spark.read.parquet(file_path)

In [ ]:
# 2. Download Data (Jan 2024 HVFHV)
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet"
output_dir = "/home/jovyan/work/data"
file_path = f"{output_dir}/fhvhv_2024_01.parquet"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

if not os.path.exists(file_path):
    print("Downloading data... this might take a minute.")
    r = requests.get(url)
    with open(file_path, 'wb') as f:
        f.write(r.content)
    print("Download complete!")

# 3. Load and Clean
df = spark.read.parquet(file_path)

# Filter for relevant columns to save RAM
# PULocationID is the pickup zone, request_datetime is the demand signal
df_filtered = df.select("hvfhs_license_num", "PULocationID", "request_datetime") \
                .filter(df.PULocationID.isNotNull())

# 4. Create "Hourly Demand" Features
# We truncate the time to the hour to see how many people requested rides at that time
df_hourly = df_filtered.withColumn("pickup_hour", F.date_trunc("hour", F.col("request_datetime"))) \
                       .groupBy("PULocationID", "pickup_hour") \
                       .count() \
                       .withColumnRenamed("count", "demand_surge")

df_hourly.show(10)